In [2]:
import torch
import yaml
import networkx as nx
from jinja2 import Template

In [3]:
with open("../workloads/gpt3_175B.yaml") as f:
    text = f.read()

template = Template(text)

# # You can override these if you want
rendered = template.render()

workload = yaml.safe_load(rendered)

einsums = workload["workload"]["einsums"]

In [4]:
G = nx.DiGraph()

def parse_einsum(expr):
    lhs, rhs = expr.split("=")
    output = lhs.split("[")[0].strip()
    inputs = [x.split("[")[0].strip() for x in rhs.split("*")]
    return output, inputs

for item in einsums:
    if isinstance(item, dict):
        expr = item["einsum"]
    else:
        expr = item

    out, ins = parse_einsum(expr)

    for i in ins:
        G.add_edge(i, out)

In [5]:
execution_order = list(nx.topological_sort(G))
print(execution_order)

['I_in', 'WV', 'WK', 'WQ', 'WZ', 'WFFA', 'WFFB', 'I', 'V', 'K', 'Q', 'QK', 'QK_softmax', 'AV', 'Z', 'FFA', 'FFB']


In [9]:
tensors = {}

# initialize inputs
B = 1
M = 128
P = 128
H = 8
E = 32
F = 32
D = H * E
G = H * E
C = 4 * G
J = G

tensors['I_in'] = torch.randn(B, M, D)

tensors['WV'] = torch.randn(H, E, D)
tensors['WK'] = torch.randn(H, E, D)
tensors['WQ'] = torch.randn(H, E, D)

tensors['WFFA'] = torch.randn(G, C)
tensors['WFFB'] = torch.randn(C, J)

In [10]:
def run_op(expr):
    lhs, rhs = expr.split("=")
    out_name = lhs.split("[")[0].strip()

    inputs = [x.strip() for x in rhs.split("*")]
    input_names = [x.split("[")[0] for x in inputs]

    # This is simplified — you'd translate to einsum notation properly
    result = tensors[input_names[0]]
    for name in input_names[1:]:
        result = torch.matmul(result, tensors[name])  # placeholder

    tensors[out_name] = result

for item in einsums:
    expr = item["einsum"] if isinstance(item, dict) else item
    run_op(expr)

RuntimeError: Expected size for first two dimensions of batch2 tensor to be: [8, 256] but got: [8, 32].